# The tradeSeq workflow

Python port of the canonical R vignette `vignettes/tradeSeq.Rmd`.
`tradeSeq` fits a negative-binomial generalized additive model (NB-GAM)
to every gene along each trajectory lineage and exposes a battery of
Wald-style tests for differential expression.

This notebook reproduces the analysis on the bundled Paul-2015 myeloid
progenitor dataset. Trajectory artifacts (`pseudotime`, `cell_weights`,
principal curves) were computed once on the R side via slingshot 2.8.0
and frozen into `tradeseq/resources/paul15_tradeseq.h5ad`; the Python
package therefore depends on no trajectory-inference library at
runtime.


## Installation

```bash
pip install tradeSeq-python
```


## Load data

In [ ]:
import numpy as np
import pandas as pd

import tradeseq as ts

adata = ts.load_paul15()
adata


The R `crv`, `countMatrix`, and `celltype` datasets are packed into a
single AnnData:

- `adata.layers["counts"]` — raw count matrix (`cells × genes`)
- `adata.obs["cluster"]`, `adata.obs["celltype"]` — categorical metadata
- `adata.obsm["X_umap"]` — 2-D UMAP
- `adata.obsm["pseudotime"]` — `slingPseudotime(crv, na=FALSE)`
- `adata.obsm["cell_weights"]` — `slingCurveWeights(crv)`
- `adata.uns["slingshot"]["curves"]` — per-lineage `s, ord, lambda`

Per the scverse mapping (`tradeseq_porting_essential_suggestions.md` §1),
the five R accessor functions (`slingPseudotime`, `slingCurveWeights`,
`slingClusterLabels`, `slingCurves`, `slingReducedDim`) are not ported as
Python symbols — users read the AnnData slots directly.


## Fit negative binomial model

Choosing the number of knots is done with `evaluate_k`. It is slow on a
full dataset, so the R vignette only includes a reference plot. Here we
demonstrate it on 20 genes × `k = 3..5`:


In [ ]:
aic_mat = ts.evaluate_k(
    adata,
    k_range=range(3, 6),
    n_genes=20,
    plot=False,
    random_state=5,
)
aic_mat[:5]


Following the R vignette, we choose `n_knots = 6`. We fit 50 genes to
keep the notebook fast — for production analysis, fit all genes.


In [ ]:
# For reproducibility.
rng = np.random.default_rng(7)
probs = adata.obsm['cell_weights'] / adata.obsm['cell_weights'].sum(axis=1, keepdims=True)
w_samp = np.zeros_like(adata.obsm['cell_weights'], dtype=np.int64)
for i in range(adata.n_obs):
    w_samp[i] = rng.multinomial(1, probs[i])

a = adata[:, :50].copy()
ts.fit_gam(a, n_knots=6, verbose=False, _w_samp=w_samp)
a


Some genes may not converge — typically due to extreme sparsity. The
convergence flag is stored in `adata.var["tradeseq_converged"]`:


In [ ]:
a.var['tradeseq_converged'].value_counts()


Knot positions are stored in `adata.uns["tradeseq"]["knots"]`:


In [ ]:
ts.nknots(a)


# Within-lineage comparisons

## Association of gene expression with pseudotime

`association_test` tests the null hypothesis that all smoother
coefficients are equal — i.e. that average gene expression is constant
along pseudotime within a lineage.


In [ ]:
asso_res = ts.association_test(a)
asso_res.head()


## Discovering progenitor marker genes

`start_vs_end_test` performs a multivariate Wald test comparing the
smoother value at the start of a lineage to the value at the end.


In [ ]:
start_res = ts.start_vs_end_test(a)
start_res.head()


Visualise the estimated smoothers for the 3rd most significant gene:


In [ ]:
o_start = start_res['waldStat'].sort_values(ascending=False).index
sig_gene_start = o_start[2]
print('sig_gene_start =', sig_gene_start)
ts.plot_smoothers(a, gene=sig_gene_start)


And colour cells in UMAP space by that gene's expression:

In [ ]:
ts.plot_gene_count(a, gene=sig_gene_start)


## Comparing specific pseudotime values within a lineage

`start_vs_end_test` can compare any two pseudotime values, not just the
endpoints. To check expression at `t = 0.1` vs `t = 0.8`:


In [ ]:
custom_res = ts.start_vs_end_test(a, pseudotime_values=(0.1, 0.8))
custom_res.head()


# Between-lineage comparisons

## Discovering differentiated cell type markers

`diff_end_test` tests whether the average expression at the endpoints
is equal between lineages.


In [ ]:
end_res = ts.diff_end_test(a)
end_res.head()


In [ ]:
o_end = end_res['waldStat'].sort_values(ascending=False).index
sig_gene_end = o_end[0]
print('sig_gene_end =', sig_gene_end)
ts.plot_smoothers(a, gene=sig_gene_end)


In [ ]:
ts.plot_gene_count(a, gene=sig_gene_end)


## Discovering genes with different expression patterns

`pattern_test` tests whether smoothed expression curves are equal
between lineages at `2 * n_knots` equally-spaced pseudotime points.


In [ ]:
pattern_res = ts.pattern_test(a)
o_pat = pattern_res['waldStat'].sort_values(ascending=False).index
print(list(o_pat[:6]))
ts.plot_smoothers(a, gene=o_pat[3])


In [ ]:
ts.plot_gene_count(a, gene=o_pat[3])


`pattern_test` (and `early_de_test`) takes an `eigen_thresh` keyword
controlling the rank threshold of the contrast's covariance matrix.
Lower values are more lenient; the default `1e-2` favours numerical
stability over inclusivity.


### Example on combining `pattern_test` with `diff_end_test` results

Genes that have a different expression *pattern* between lineages but
no different *endpoint* expression are the most interesting transient-
expression candidates. Score them by a sum-of-squares of the two
Wald-statistic ranks:


In [ ]:
import ggplot2_py as g

compare = (
    pattern_res.rename(columns={'waldStat': 'pattern'})[['pattern']]
    .join(end_res.rename(columns={'waldStat': 'end'})[['end']], how='inner')
)
compare['transient_score'] = (
    (-compare['end']).rank(method='min').astype(int) ** 2
    + compare['pattern'].rank(method='min').astype(int) ** 2
)
compare['log_pattern'] = np.log(compare['pattern'].clip(lower=1e-3))
compare['log_end'] = np.log(compare['end'].clip(lower=1e-3))
(
    g.ggplot(compare, g.aes(x='log_pattern', y='log_end'))
    + g.geom_point(g.aes(colour='transient_score'))
    + g.labs(x='pattern_test Wald (log)', y='diff_end_test Wald (log)')
    + g.scale_color_continuous(low='yellow', high='red')
    + g.theme_classic()
)


In [ ]:
top_transient = compare['transient_score'].idxmax()
print('top_transient =', top_transient)
ts.plot_smoothers(a, gene=top_transient)


In [ ]:
ts.plot_gene_count(a, gene=top_transient)


The top 5 transient-expression candidates by this ranking:


In [ ]:
compare.sort_values('transient_score', ascending=False).head(5).index.tolist()


If your dataset contains the classic Irf8 progenitor marker, this is
the place where the R vignette inspects it. Try:


In [ ]:
if 'Irf8' in a.var_names:
    display(ts.plot_smoothers(a, gene='Irf8'))
    display(ts.plot_gene_count(a, gene='Irf8'))


## Early drivers of differentiation

`early_de_test(adata, knots=(k_low, k_high))` tests whether the
smoothed expression curves are equal between lineages in the pseudotime
region bounded by knots `k_low` and `k_high`. Setting `knots=(1, 2)`
probes early differentiation right after the bifurcation.


As in the R vignette, first show the fitted knot positions on the
trajectory, coloured by cluster assignment:


In [ ]:
ts.plot_gene_count(a, clusters_key='cluster')


In [ ]:
early_de_res = ts.early_de_test(a, knots=(1, 2))
o_early = early_de_res['waldStat'].sort_values(ascending=False).index
print(list(o_early[:6]))
ts.plot_smoothers(a, gene=o_early[1])


In [ ]:
ts.plot_gene_count(a, gene=o_early[1])


# Differential expression in large datasets

Large datasets can render statistically significant fold changes that
are biologically inconsequential. Every Wald test accepts an `l2fc`
argument that specifies the absolute log2-fold-change cut-off; only
genes whose effect exceeds this cut-off are reported as significant.


In [ ]:
# Testing against a fold-change threshold of 2.
start2 = ts.start_vs_end_test(a, l2fc=np.log2(2))
# Testing against a fold-change threshold of 1.5.
pat2 = ts.pattern_test(a, l2fc=np.log2(1.5))
start2.head(), pat2.head()


# Clustering of genes according to their expression pattern

## Extracting fitted values to use with any clustering method

Use `predict_cells` to get fitted means per cell (count scale) and
`predict_smooth` for a uniform grid along each lineage. Both are
suitable inputs for any clustering method.


In [ ]:
yhat = ts.predict_cells(a, gene='Irf8') if 'Irf8' in a.var_names else ts.predict_cells(a, gene=str(a.var_names[0]))
yhat.shape


In [ ]:
y_smooth = ts.predict_smooth(
    a, gene=('Irf8' if 'Irf8' in a.var_names else str(a.var_names[0])),
    n_points=40,
)
y_smooth.head()


## Clustering using RSEC, clusterExperiment

The R package uses `clusterExperiment::RSEC` (resampling-based
sequential ensemble clustering); the Python port substitutes
`sklearn.cluster.AgglomerativeClustering` with silhouette-based `k`
selection. The backend swap is a documented Tier-3 deviation.


In [ ]:
clus_res = ts.cluster_expression_patterns(
    a, n_points=20, genes=list(a.var_names[:50]),
)
# ClusterResult is a NamedTuple mirroring R's `list(rsec, yhatScaled)`:
# `.cluster_labels` is a pandas.Series indexed by gene, `.yhat_scaled` is
# the (n_genes, n_lineages * n_points) row-standardised matrix, and
# `.long_df` is a tidy view useful for ggplot.
clus_df = clus_res.long_df
clus_df.head()


Visualise the first four clusters' normalized expression curves:


In [ ]:
import ggplot2_py as g
import patchwork

# `time_point` is encoded as 'l<lineage>:t<pseudotime>' -- split it.
tp = clus_df['time_point'].astype(str).str.split(':', n=1, expand=True)
clus_df = clus_df.assign(
    lineage=tp[0],
    pseudotime=tp[1].str.replace('t', '', regex=False).astype(float),
)

plots = []
for k in sorted(clus_df['cluster'].unique())[:4]:
    sub = clus_df[clus_df['cluster'] == k].copy()
    sub['gene_lineage'] = sub['gene'].astype(str) + '_' + sub['lineage'].astype(str)
    range_df = pd.DataFrame({
        'pseudotime': [sub['pseudotime'].min(), sub['pseudotime'].max()],
        'profile': [sub['profile'].min(), sub['profile'].max()],
    })
    plots.append(
        g.ggplot(sub, g.aes(x='pseudotime', y='profile',
                            group='gene_lineage', colour='lineage'))
        + g.geom_blank(data=range_df,
                       mapping=g.aes(x='pseudotime', y='profile'),
                       inherit_aes=False)
        + g.geom_line(linewidth=1.5, show_legend=False)
        + g.scale_color_manual(values=['#FFA500', '#9BCD9B'],
                               breaks=['l1', 'l2'])
        + g.labs(title=f'Cluster {k}',
                 x='Pseudotime', y='Normalized expression')
        + g.theme_classic()
    )
patchwork.wrap_plots(plots, nrow=1)


# Wrap-up

We've walked through tradeSeq's full Wald battery:

- **Foundation**: `fit_gam`, `nknots`, `evaluate_k`,
  `plot_evaluatek_results`
- **Within-lineage**: `association_test`, `start_vs_end_test`
- **Between-lineage**: `diff_end_test`, `pattern_test`,
  `early_de_test`, `condition_test`
- **Prediction**: `predict_cells`, `predict_smooth`
- **Visualization**: `plot_smoothers`, `plot_gene_count`
- **Downstream**: `cluster_expression_patterns`,
  `get_smoother_pvalues`, `get_smoother_test_stats`

The accompanying `fitGAM.ipynb` notebook goes deeper into model-fitting
options (covariates, parallelism, list-mode output, convergence
diagnostics).


# Contributing and requesting

The Python port keeps the same extension point as the R package: novel
hypothesis tests should be built on the fitted coefficient, covariance,
design-matrix and prediction helpers exposed by `fit_gam`,
`predict_gam`, and the Wald-test machinery. File issues or proposed
tests in the project repository so they can be evaluated against the R
gold standard before becoming public API.


# Session

In [ ]:
import platform, sys
print('python:', sys.version.split()[0])
print('platform:', platform.platform())
print('tradeseq:', ts.__version__)


# References

This notebook is the Python/scverse counterpart of
`tradeSeq/vignettes/tradeSeq.Rmd`; see the original vignette bibliography
for the statistical model, slingshot, clusterExperiment, and Paul-2015
citations.
